# Cross-Dataset Phishing Website Detection
## Notebook 4: Feature Selection, Ablation Study & Hyperparameter Tuning

**Objectives:**
1. Compute **Mutual Information (MI)** scores for PhiUSIIL features to quantify predictive signals.
2. Conduct an **Ablation Study** by dropping top deterministic features to analyze whether near-100% accuracy is caused by single shortcuts or redundant features.
3. Perform **Hyperparameter Tuning** via `RandomizedSearchCV` on aligned features to optimize generalization.


In [ ]:
# Environment & Path Setup (supports local execution and Google Colab)
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/phishing-project')
except ImportError:
    BASE = Path('..').resolve() if Path('.').resolve().name == 'notebooks' else Path('.').resolve()

sys.path.insert(0, str(BASE))
print(f"Project base path: {BASE}")

from src.data_utils import project_paths
dirs = project_paths(BASE)
PROC_DIR = dirs['processed']
TAB_DIR = dirs['tables']
MOD_DIR = dirs['models']
FIG_DIR = dirs['figures']


### 1. Mutual Information Analysis
Mutual Information measures the reduction in entropy of the target label given each feature.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

mi_df = pd.read_csv(TAB_DIR / '04_phiusiil_mutual_info.csv')
print("Top 10 Most Informative Features in PhiUSIIL:")
display(mi_df.head(10))

plt.figure(figsize=(10, 5))
sns.barplot(data=mi_df.head(15), x='mutual_info', y='feature', palette='Blues_r')
plt.title("Top 15 Features by Mutual Information (PhiUSIIL)")
plt.xlabel("Mutual Information Score")
plt.show()


### 2. Feature Ablation Study
We evaluate Random Forest after removing top deterministic features (`URLSimilarityIndex`, `URLCharProb`, `TLDLegitimateProb`).


In [ ]:
ablation_df = pd.read_csv(TAB_DIR / '04_phiusiil_ablation_results.csv')
display(ablation_df)


**Ablation Takeaway:**
Dropping the top 3 features results in a negligible drop (99.998% to 99.955% accuracy). This confirms that PhiUSIIL possesses multiple redundant, deterministic features rather than a single procedural flaw.


### 3. Hyperparameter Tuning on Aligned Features
Optimizing Random Forest and XGBoost on UCI aligned data using Stratified 5-Fold Cross-Validation.


In [ ]:
tuning_df = pd.read_csv(TAB_DIR / '04_tuning_summary.csv')
display(tuning_df)


### 4. Summary of Optimal Hyperparameters
- **Random Forest:** `n_estimators=200, max_depth=3, min_samples_split=2, min_samples_leaf=2` ($F_1 = 0.882$).
- **XGBoost:** `n_estimators=200, max_depth=3, learning_rate=0.2, subsample=0.8` ($F_1 = 0.883$).
Constraining tree depth to 3 prevents overfitting while maximizing cross-validation performance.
